In [8]:
import json
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

with open("/content/drive/MyDrive/results_100.json", "r") as f:
    results = json.load(f)

print(f"Reloaded {len(results)} results")

Mounted at /content/drive
Reloaded 100 results


In [9]:
!pip install transformers accelerate bitsandbytes datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.5 MB/s eta 0:00:00


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit quantization config — loads the model in compressed form
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"          # automatically puts layers on GPU
)

print("Model loaded!")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded!


In [11]:
from datasets import load_dataset

dataset = load_dataset("ccdv/arxiv-summarization", split="test[:10]")

print(f"Loaded {len(dataset)} examples")
print("\n--- Example article (first 300 chars) ---")
print(dataset[0]["article"][:300])
print("\n--- Reference abstract ---")
print(dataset[0]["abstract"])

README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00003-of-00015.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Loaded 10 examples

--- Example article (first 300 chars) ---
for about 20 years the problem of properties of short - term changes of solar activity has been considered extensively . 
 many investigators studied the short - term periodicities of the various indices of solar activity . 
 several periodicities were detected , but the periodicities about 155 days

--- Reference abstract ---
the short - term periodicities of the daily sunspot area fluctuations from august 1923 to october 1933 are discussed . for these data 
 the correlative analysis indicates negative correlation for the periodicity of about @xmath0 days , but the power spectrum analysis indicates a statistically significant peak in this time interval . 
 a new method of the diagnosis of an echo - effect in spectrum is proposed and it is stated that the 155-day periodicity is a harmonic of the periodicities from the interval of @xmath1 $ ] days .    the autocorrelation functions for the daily sunspot area fluctuations and 

In [12]:
#cell 3b


import re

raw = dataset[0]["article"]

# Find all citation-like patterns
found = re.findall(r'[\\@]?[a-z]*cite[a-z]*[^a-z]', raw[:5000], re.IGNORECASE)
print("Citation patterns found:", set(found))

# Show raw text around "xcite"
for match in re.finditer(r'.{20}xcite.{20}', raw, re.IGNORECASE):
    print(repr(match.group()))


def clean_text(text):
    """Remove LaTeX math placeholders and stray brackets that confuse the model."""

    # Remove LaTeX citation commands
    text = re.sub(r'@xcite\s*', '', text)          # handles @xcite (this dataset's format)
    text = re.sub(r'\\xcite\{[^}]*\}', '', text)   # handles \xcite{...} (standard LaTeX)
    text = re.sub(r'\\cite\{[^}]*\}', '', text)    # handles \cite{...} (standard LaTeX)
    # Remove figure/table markup
    text = re.sub(r',title="[^"]*"', '', text)
    text = re.sub(r'width=\d+', '', text)
    # Remove @xmath tokens
    text = re.sub(r'@xmath\d+', '', text)
    # Remove lone brackets
    text = re.sub(r'\s*[\[\]]\s*', ' ', text)
    # Remove lines that are more than 30% math-like symbols
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        math_chars = len(re.findall(r'[@\^_{}\\=+<>|]', line))
        if len(line) > 0 and math_chars / len(line) < 0.3:
            clean_lines.append(line)
    text = '\n'.join(clean_lines)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_introduction(text, max_words=2000):
    """
    Extract the introduction section, or fall back to the first clean paragraph.
    """

    # First clean the text
    text = clean_text(text)
    lower = text.lower()

    # Try to find introduction heading
    intro_patterns = [
        r'\bintroduction\b',
        r'\b1\.\s*introduction\b',
        r'\bi\.\s*introduction\b',
    ]

    intro_start = None
    for pattern in intro_patterns:
        match = re.search(pattern, lower)
        if match:
            intro_start = match.start()
            break

    if intro_start is not None:
        text = text[intro_start:]
    # If no intro found, we still use cleaned full text from the start

    return " ".join(text.split()[:max_words])

Citation patterns found: {'@xcite '}
' was discovered by @xcite in the occurence ra'
'r cycles 19 and 20 @xcite , but it was not fo'
'unspot area data . @xcite studied the sunspot'
'swerepresentedby + @xcite , but for sunspot n'
'ed noise of markov @xcite for the choice of t'
'nce level @xmath12 @xcite to verify a statist'
'ious solar indices @xcite , the sunspot activ'
'aily sunspot areas @xcite .    for the southe'
' time series ( see @xcite , figures 7 a - c )'
'e also obtained by @xcite for daily sunspot n'


In [13]:
def generate_summary(text, examples=None, max_input_words=3000, max_new_tokens=400):

    if examples is None:
        # Zero-shot: we have plenty of room
        truncated = extract_introduction(text, max_words=max_input_words)
        messages = [
            {
                "role": "user",
                "content": f"""You are a scientific paper summarizer. Read the following paper and write a concise abstract of at most 300 words. Output only the abstract text, nothing else. Do not copy or continue the paper text.

Paper:
{truncated}

Abstract:"""
            }
        ]
    else:
        # Few-shot: budget is tight — reduce everything
        # Total budget: 4096 - 400 (generation) - 200 (overhead) = ~3500 tokens
        # Split: ~800 per few-shot example article, ~600 for test paper
        truncated = extract_introduction(text, max_words=600)

        messages = [
            {
                "role": "user",
                "content": "You are a scientific paper summarizer. I will show you some papers and you will write a concise abstract for each one. Output only the abstract text, nothing else."
            },
            {
                "role": "assistant",
                "content": "Understood. Please share the papers and I will write a concise abstract for each one."
            }
        ]

        for ex_article, ex_abstract in examples:
            ex_truncated = " ".join(clean_text(ex_article).split()[:200])
            messages.append({
                "role": "user",
                "content": f"Paper:\n{ex_truncated}\n\nWrite the abstract:"
            })
            messages.append({
                "role": "assistant",
                "content": ex_abstract.strip()
            })

        messages.append({
            "role": "user",
            "content": f"Paper:\n{truncated}\n\nWrite the abstract:"
        })

    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Safety check — print token count on first call
    token_count = len(tokenizer.encode(prompt))
    if token_count > 4096:
        print(f"WARNING: prompt is {token_count} tokens — still over limit")

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return summary.strip()

In [15]:
for i in [0, 25, 75]:
    article = test_dataset[i]["article"]
    few = generate_summary(article, examples=few_shot_examples)
    print(f"\n{'='*60}")
    print(f"EXAMPLE {i} FEW-SHOT:\n{few[:300]}")

NameError: name 'test_dataset' is not defined

In [16]:
test_article = test_dataset[0]["article"]
truncated = extract_introduction(test_article, max_words=600)  # updated

messages = [
    {"role": "user", "content": "You are a scientific paper summarizer. I will show you some papers and you will write a concise abstract for each one. Output only the abstract text, nothing else."},
    {"role": "assistant", "content": "Understood. Please share the papers and I will write a concise abstract for each one."}
]
for ex_article, ex_abstract in few_shot_examples:
    ex_truncated = " ".join(clean_text(ex_article).split()[:200])  # updated
    messages.append({"role": "user", "content": f"Paper:\n{ex_truncated}\n\nWrite the abstract:"})
    messages.append({"role": "assistant", "content": ex_abstract.strip()})
messages.append({"role": "user", "content": f"Paper:\n{truncated}\n\nWrite the abstract:"})

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
token_count = len(tokenizer.encode(prompt))
print(f"Total tokens: {token_count}")
print(f"Tokens left for generation: {4096 - token_count}")

NameError: name 'test_dataset' is not defined

In [17]:
for i in range(3):
    article = dataset[i]["article"]
    reference = dataset[i]["abstract"]

    # Show what the model will actually receive
    preview = extract_introduction(article, max_words=2000)
    print(f"\n{'='*60}")
    print(f"PAPER {i+1}")
    print(f"\n[INPUT PREVIEW - first 200 chars fed to model]\n{preview[:200]}")

    generated = generate_summary(article)

    print(f"\n[REFERENCE ABSTRACT]\n{reference[:400]}")
    print(f"\n[GENERATED SUMMARY]\n{generated[:400]}")


PAPER 1

[INPUT PREVIEW - first 200 chars fed to model]
for about 20 years the problem of properties of short - term changes of solar activity has been considered extensively . many investigators studied the short - term periodicities of the various indice

[REFERENCE ABSTRACT]
the short - term periodicities of the daily sunspot area fluctuations from august 1923 to october 1933 are discussed . for these data 
 the correlative analysis indicates negative correlation for the periodicity of about @xmath0 days , but the power spectrum analysis indicates a statistically significant peak in this time interval . 
 a new method of the diagnosis of an echo - effect in spectrum i

[GENERATED SUMMARY]
This paper explores the short-term changes in solar activity, particularly focusing on the 155-day periodicity, which is a commonly detected pattern in various solar activity indices. The periodicity was initially discovered in the occurrence rate of gamma-ray flares by the Gamma-Ray Spectrometer 

In [18]:
val_dataset = load_dataset("ccdv/arxiv-summarization", split="validation[:50]")

for i in range(20):
    article_preview = " ".join(clean_text(val_dataset[i]["article"]).split()[:60])
    abstract = val_dataset[i]["abstract"][:300]
    print(f"\n--- Example {i} ---")
    print(f"Article start: {article_preview}")
    print(f"Abstract: {abstract}")
    print()


--- Example 0 ---
Article start: the interest in anchoring phenomena and phenomena in confined nematic liquid crystals has largely been driven by their potential use in liquid crystal display devices . the twisted nematic liquid crystal cell serves as an example . it consists of a nematic liquid crystal confined between two parallel walls , both providing homogeneous planar anchoring but with mutually perpendicular easy
Abstract: we study the phase behavior of a nematic liquid crystal confined between a flat substrate with strong anchoring and a patterned substrate whose structure and local anchoring strength we vary . by first evaluating an effective surface free energy function characterizing the patterned substrate we der


--- Example 1 ---
Article start: galaxy clusters , as the largest peaks in the cosmic density field , play an important role in astrophysics and cosmology ( e.g. * ? ? ? * ; * ? ? ? structure formation theory , realized in large scale n - body simulations , make

In [19]:
# Load the two chosen examples from the validation set
val_dataset = load_dataset("ccdv/arxiv-summarization", split="validation[:10]")

few_shot_examples = [
    (val_dataset[4]["article"], val_dataset[4]["abstract"]),  # interdisciplinary research
    (val_dataset[8]["article"], val_dataset[8]["abstract"]),  # molecular motors / TASEP
]

# Verify they look clean
for i, (article, abstract) in enumerate(few_shot_examples):
    cleaned = extract_introduction(article, max_words=500)
    print(f"\n--- Few-shot example {i+1} ---")
    print(f"Article (first 200 chars): {cleaned[:200]}")
    print(f"Abstract: {abstract[:300]}")


--- Few-shot example 1 ---
Article (first 200 chars): interdisciplinary research has recently gained a central role in the advancement of science , leading to important achievements . for instance , the 2014 nobel prize in chemistry was awarded to two ph
Abstract: nowadays , scientific challenges usually require approaches that cross traditional boundaries between academic disciplines , driving many researchers towards interdisciplinarity . despite its obvious importance , there is a lack of studies on how to quantify the influence of interdisciplinarity on t

--- Few-shot example 2 ---
Article (first 200 chars): recently , driven diffusive systems have attracted the interests of physicists because the systems show surprisingly rich and complex behavior.as a simple model of driven diffusive systems , asymmetri
Abstract: in this paper , we study a two - lane totally asymmetric simple exclusion process ( tasep ) coupled with random attachment and detachment of particles ( langmuir kinet

In [4]:
import json
from tqdm import tqdm

# Load 100 test examples
test_dataset = load_dataset("ccdv/arxiv-summarization", split="test[:100]")
print(f"Loaded {len(test_dataset)} test examples")

results = []

for i, example in enumerate(tqdm(test_dataset, desc="Running inference")):
    article = example["article"]
    reference = example["abstract"]

    # Zero-shot
    try:
        zero_shot = generate_summary(article, examples=None)
    except Exception as e:
        zero_shot = ""
        print(f"Zero-shot error on example {i}: {e}")

    # Few-shot
    try:
        few_shot = generate_summary(article, examples=few_shot_examples)
    except Exception as e:
        few_shot = ""
        print(f"Few-shot error on example {i}: {e}")

    results.append({
        "id": i,
        "reference": reference,
        "zero_shot": zero_shot,
        "few_shot": few_shot
    })

    # Checkpoint every 25 examples
    if i % 25 == 0 and i > 0:
        with open("results_checkpoint.json", "w") as f:
            json.dump(results, f)
        print(f"\nCheckpoint saved at example {i}")

# Final save
with open("results_100.json", "w") as f:
    json.dump(results, f)

print("\nDone! Saved results_100.json")
print(f"Total examples: {len(results)}")

NameError: name 'load_dataset' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("results_100.json", "/content/drive/MyDrive/results_100.json")
print("Saved to Google Drive")

In [20]:
import json

with open("results_100.json", "r") as f:
    results = json.load(f)

print(f"Total results: {len(results)}")
print(f"\nKeys in each result: {list(results[0].keys())}")

# Spot check 3 examples
for i in [0, 25, 75]:
    print(f"\n{'='*60}")
    print(f"EXAMPLE {i}")
    print(f"\nReference (first 200): {results[i]['reference'][:200]}")
    print(f"\nZero-shot (first 200): {results[i]['zero_shot'][:200]}")
    print(f"\nFew-shot (first 200):  {results[i]['few_shot'][:200]}")



empty_zero = [r["id"] for r in results if r["zero_shot"] == ""]
empty_few  = [r["id"] for r in results if r["few_shot"] == ""]

print(f"Empty zero-shot outputs: {len(empty_zero)} — {empty_zero}")
print(f"Empty few-shot outputs:  {len(empty_few)} — {empty_few}")

FileNotFoundError: [Errno 2] No such file or directory: 'results_100.json'

In [ ]:
!pip install rouge-score bert-score summac -q

In [28]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

zero_scores = {"rouge1": [], "rouge2": [], "rougeL": []}
few_scores  = {"rouge1": [], "rouge2": [], "rougeL": []}

for r in results:
    ref = r["reference"]

    z = scorer.score(ref, r["zero_shot"])
    f = scorer.score(ref, r["few_shot"])

    for metric in ["rouge1", "rouge2", "rougeL"]:
        zero_scores[metric].append(z[metric].fmeasure)
        few_scores[metric].append(f[metric].fmeasure)

print("=" * 45)
print(f"{'Metric':<12} {'Zero-shot':>12} {'Few-shot':>12}")
print("=" * 45)
for metric in ["rouge1", "rouge2", "rougeL"]:
    z_mean = np.mean(zero_scores[metric])
    f_mean = np.mean(few_scores[metric])
    print(f"{metric:<12} {z_mean:>12.4f} {f_mean:>12.4f}")
print("=" * 45)

Metric          Zero-shot     Few-shot
rouge1             0.4018       0.4154
rouge2             0.1449       0.1405
rougeL             0.2140       0.2253


In [ ]:
!pip install evaluate absl-py rouge-score -q

In [ ]:
!pip install transformers==4.44.0 -q --force-reinstall

In [22]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

print("Loading sentence-transformers model...")
st_model = SentenceTransformer("all-MiniLM-L6-v2")

refs      = [r["reference"] for r in results]
zero_outs = [r["zero_shot"] for r in results]
few_outs  = [r["few_shot"]  for r in results]

print("Encoding references...")
ref_embs  = st_model.encode(refs,      convert_to_tensor=True, show_progress_bar=True)
print("Encoding zero-shot outputs...")
zero_embs = st_model.encode(zero_outs, convert_to_tensor=True, show_progress_bar=True)
print("Encoding few-shot outputs...")
few_embs  = st_model.encode(few_outs,  convert_to_tensor=True, show_progress_bar=True)

zero_sim = util.cos_sim(ref_embs, zero_embs).diagonal().cpu().numpy()
few_sim  = util.cos_sim(ref_embs, few_embs).diagonal().cpu().numpy()

print("\n" + "=" * 45)
print(f"{'Metric':<25} {'Zero-shot':>8} {'Few-shot':>8}")
print("=" * 45)
print(f"{'Semantic Similarity':<25} {zero_sim.mean():>8.4f} {few_sim.mean():>8.4f}")
print("=" * 45)

# Save scores
import json
sem_scores = {
    "zero_shot": zero_sim.tolist(),
    "few_shot":  few_sim.tolist()
}
with open("/content/drive/MyDrive/sem_scores.json", "w") as f:
    json.dump(sem_scores, f)
print("Saved to Drive")

Loading sentence-transformers model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding zero-shot outputs...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Encoding few-shot outputs...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Metric                    Zero-shot Few-shot
Semantic Similarity         0.7440   0.7342
Saved to Drive


In [24]:
from transformers import pipeline
import numpy as np

print("Loading NLI model...")
nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=0
)

refs      = [r["reference"] for r in results]
zero_outs = [r["zero_shot"] for r in results]
few_outs  = [r["few_shot"]  for r in results]

def consistency_score(sources, summaries, batch_size=8):
    scores = []
    for i in range(0, len(sources), batch_size):
        batch_sources   = sources[i:i+batch_size]
        batch_summaries = summaries[i:i+batch_size]
        pairs = [
            {"text": src[:1000], "text_pair": summ[:500]}
            for src, summ in zip(batch_sources, batch_summaries)
        ]
        preds = nli(pairs, truncation=True)
        for pred in preds:
            if pred["label"] == "ENTAILMENT":
                scores.append(pred["score"])
            elif pred["label"] == "CONTRADICTION":
                scores.append(1 - pred["score"])
            else:
                scores.append(0.5)
    return np.array(scores)

print("Scoring zero-shot consistency...")
zero_consistency = consistency_score(refs, zero_outs)

print("Scoring few-shot consistency...")
few_consistency  = consistency_score(refs, few_outs)

print("\n" + "=" * 45)
print(f"{'Metric':<25} {'Zero-shot':>8} {'Few-shot':>8}")
print("=" * 45)
print(f"{'NLI Consistency':<25} {zero_consistency.mean():>8.4f} {few_consistency.mean():>8.4f}")
print("=" * 45)

# Save scores
consistency_scores = {
    "zero_shot": zero_consistency.tolist(),
    "few_shot":  few_consistency.tolist()
}
with open("/content/drive/MyDrive/consistency_scores.json", "w") as f:
    import json
    json.dump(consistency_scores, f)
print("Saved to Drive")

Loading NLI model...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring zero-shot consistency...
Scoring few-shot consistency...

Metric                    Zero-shot Few-shot
NLI Consistency             0.5000   0.5000
Saved to Drive


In [25]:
# Check what labels the model actually outputs
test_pairs = [
    {"text": refs[0][:1000], "text_pair": zero_outs[0][:500]}
]
test_pred = nli(test_pairs, truncation=True)
print("Raw prediction:", test_pred)
print("Label:", test_pred[0]["label"])
print("Score:", test_pred[0]["score"])

Raw prediction: [{'label': 'neutral', 'score': 0.9308363795280457}]
Label: neutral
Score: 0.9308363795280457


In [26]:
import numpy as np
from transformers import pipeline

print("Loading NLI model...")
nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=0
)

# Fix labels to lowercase
def get_consistency(sources, summaries, batch_size=8):
    scores = []
    for i in range(0, len(sources), batch_size):
        batch_src = sources[i:i+batch_size]
        batch_sum = summaries[i:i+batch_size]

        pairs = [
            {"text": src[:1500], "text_pair": summ[:500]}
            for src, summ in zip(batch_src, batch_sum)
        ]
        preds = nli(pairs, truncation=True)

        for pred in preds:
            label = pred["label"].lower()  # fix: lowercase comparison
            if label == "entailment":
                scores.append(pred["score"])
            elif label == "contradiction":
                scores.append(1 - pred["score"])
            else:  # neutral
                scores.append(0.5)
    return np.array(scores)

# Use the actual article text as source, not the reference abstract
# Reload test dataset to get original articles
from datasets import load_dataset
test_dataset = load_dataset("ccdv/arxiv-summarization", split="test[:100]")

import re
def clean_text(text):
    text = re.sub(r'@xcite\s*', '', text)
    text = re.sub(r'\\xcite\{[^}]*\}', '', text)
    text = re.sub(r'\\cite\{[^}]*\}', '', text)
    text = re.sub(r',title="[^"]*"', '', text)
    text = re.sub(r'width=\d+', '', text)
    text = re.sub(r'@xmath\d+', '', text)
    text = re.sub(r'\s*[\[\]]\s*', ' ', text)
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        math_chars = len(re.findall(r'[@\^_{}\\=+<>|]', line))
        if len(line) > 0 and math_chars / len(line) < 0.3:
            clean_lines.append(line)
    text = '\n'.join(clean_lines)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Use cleaned article introductions as source
sources = [clean_text(ex["article"])[:1500] for ex in test_dataset]
zero_outs = [r["zero_shot"] for r in results]
few_outs  = [r["few_shot"]  for r in results]

print("Scoring zero-shot consistency...")
zero_consistency = get_consistency(sources, zero_outs)

print("Scoring few-shot consistency...")
few_consistency  = get_consistency(sources, few_outs)

print("\n" + "=" * 45)
print(f"{'Metric':<25} {'Zero-shot':>8} {'Few-shot':>8}")
print("=" * 45)
print(f"{'NLI Consistency':<25} {zero_consistency.mean():>8.4f} {few_consistency.mean():>8.4f}")
print("=" * 45)

# Quick label distribution check
test_pred = nli([{"text": sources[0], "text_pair": zero_outs[0][:500]}], truncation=True)
print(f"\nSample prediction: {test_pred}")

# Save
import json
with open("/content/drive/MyDrive/consistency_scores.json", "w") as f:
    json.dump({"zero_shot": zero_consistency.tolist(), "few_shot": few_consistency.tolist()}, f)
print("Saved to Drive")

Loading NLI model...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring zero-shot consistency...
Scoring few-shot consistency...

Metric                    Zero-shot Few-shot
NLI Consistency             0.6698   0.6723

Sample prediction: [{'label': 'entailment', 'score': 0.995262861251831}]
Saved to Drive


In [27]:
import json
import numpy as np
from rouge_score import rouge_scorer

# Recompute per-example ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

for r in results:
    ref = r["reference"]
    z = scorer.score(ref, r["zero_shot"])
    f = scorer.score(ref, r["few_shot"])
    r["rouge1_zero"] = z["rouge1"].fmeasure
    r["rouge1_few"]  = f["rouge1"].fmeasure

# Load consistency and semantic scores
with open("/content/drive/MyDrive/consistency_scores.json") as f:
    cons = json.load(f)
with open("/content/drive/MyDrive/sem_scores.json") as f:
    sem = json.load(f)

for i, r in enumerate(results):
    r["consistency_zero"] = cons["zero_shot"][i]
    r["consistency_few"]  = cons["few_shot"][i]
    r["sem_zero"]         = sem["zero_shot"][i]
    r["sem_few"]          = sem["few_shot"][i]

# Composite score: average of rouge1 + semantic + consistency
for r in results:
    r["composite_zero"] = np.mean([r["rouge1_zero"], r["sem_zero"], r["consistency_zero"]])
    r["composite_few"]  = np.mean([r["rouge1_few"],  r["sem_few"],  r["consistency_few"]])

# Sort by zero-shot composite score
sorted_results = sorted(results, key=lambda x: x["composite_zero"])

# Bottom 5 — worst performing
print("=" * 70)
print("WORST 5 EXAMPLES (by zero-shot composite score)")
print("=" * 70)
for r in sorted_results[:5]:
    print(f"\nExample ID: {r['id']}")
    print(f"  ROUGE-1:     zero={r['rouge1_zero']:.3f}  few={r['rouge1_few']:.3f}")
    print(f"  Semantic:    zero={r['sem_zero']:.3f}  few={r['sem_few']:.3f}")
    print(f"  Consistency: zero={r['consistency_zero']:.3f}  few={r['consistency_few']:.3f}")
    print(f"  Composite:   zero={r['composite_zero']:.3f}  few={r['composite_few']:.3f}")
    print(f"\n  Reference:  {r['reference'][:200]}")
    print(f"\n  Zero-shot:  {r['zero_shot'][:200]}")
    print(f"\n  Few-shot:   {r['few_shot'][:200]}")
    print("-" * 70)

# Top 5 — best performing
print("\n" + "=" * 70)
print("BEST 5 EXAMPLES (by zero-shot composite score)")
print("=" * 70)
for r in sorted_results[-5:]:
    print(f"\nExample ID: {r['id']}")
    print(f"  ROUGE-1:     zero={r['rouge1_zero']:.3f}  few={r['rouge1_few']:.3f}")
    print(f"  Semantic:    zero={r['sem_zero']:.3f}  few={r['sem_few']:.3f}")
    print(f"  Consistency: zero={r['consistency_zero']:.3f}  few={r['consistency_few']:.3f}")
    print(f"  Composite:   zero={r['composite_zero']:.3f}  few={r['composite_few']:.3f}")
    print(f"\n  Reference:  {r['reference'][:200]}")
    print(f"\n  Zero-shot:  {r['zero_shot'][:200]}")
    print(f"\n  Few-shot:   {r['few_shot'][:200]}")
    print("-" * 70)

WORST 5 EXAMPLES (by zero-shot composite score)

Example ID: 71
  ROUGE-1:     zero=0.065  few=0.372
  Semantic:    zero=-0.016  few=0.833
  Consistency: zero=0.500  few=0.995
  Composite:   zero=0.183  few=0.733

  Reference:  a ginzburg - landau model for the macroscopic behaviour of a shape memory alloy is proposed . 
 the model is one - dimensional in essence , in that we consider the effect of the martensitic phase tran

  Zero-shot:  1 ) , 119131 ( 2006 ) . j. a. shaw , smart mater . 16 , s155s169 ( 2007 ) . j. a. shaw , smart mater . 16 , s155s169 ( 2007 ) . j. a. shaw , smart mater . 16 , s155s169 ( 2007 ) . j. a. shaw , smart m

  Few-shot:   This paper reviews various constitutive models for shape memory alloys (SMA), which exhibit unique properties of shape memory and pseudoelasticity due to martensitic phase transition. The mechanical b
----------------------------------------------------------------------

Example ID: 78
  ROUGE-1:     zero=0.232  few=0.297
  Semantic:    